In [26]:
import pandas as pd
import numpy as np

In [27]:
#Load the hour dataset
df = pd.read_csv('hour.csv')

In [28]:
df['dteday'] = pd.to_datetime(df['dteday'])
df['day'] = df['dteday'].dt.day

# Create a weekend flag (0=Sunday, 6=Saturday)
df['is_weekend'] = df['weekday'].apply(lambda x: 1 if x in [0, 6] else 0)

print(df[['dteday', 'day', 'weekday', 'is_weekend']].head())

      dteday  day  weekday  is_weekend
0 2011-01-01    1        6           1
1 2011-01-01    1        6           1
2 2011-01-01    1        6           1
3 2011-01-01    1        6           1
4 2011-01-01    1        6           1


In [29]:
def encode_cyclical(df, col, max_val):
    df[col + '_sin'] = np.sin(2 * np.pi * df[col] / max_val)
    df[col + '_cos'] = np.cos(2 * np.pi * df[col] / max_val)
    return df

df = encode_cyclical(df, 'hr', 24)        # Hour of day
df = encode_cyclical(df, 'mnth', 12)      # Month of year
df = encode_cyclical(df, 'weekday', 7)    # Day of week

print(df[['hr', 'hr_sin', 'hr_cos']].head())

   hr    hr_sin    hr_cos
0   0  0.000000  1.000000
1   1  0.258819  0.965926
2   2  0.500000  0.866025
3   3  0.707107  0.707107
4   4  0.866025  0.500000


In [47]:
def encode_cyclical(df, col, max_val):

    df[col + '_sin'] = np.sin(2 * np.pi * df[col] / max_val)

    df[col + '_cos'] = np.cos(2 * np.pi * df[col] / max_val)
    return df
df = encode_cyclical(df, 'mnth', 12)

df = encode_cyclical(df, 'weekday', 7)
print(df[['mnth', 'mnth_sin', 'mnth_cos']].head()) 

   mnth  mnth_sin  mnth_cos
0     1       0.5  0.866025
1     1       0.5  0.866025
2     1       0.5  0.866025
3     1       0.5  0.866025
4     1       0.5  0.866025


In [30]:
df['temp_atemp_diff'] = df['atemp'] - df['temp']

# Capture combined effect of Heat and Humidity
df['temp_hum_interaction'] = df['temp'] * df['hum']

print(df[['temp', 'atemp', 'temp_atemp_diff', 'temp_hum_interaction']].head())

   temp   atemp  temp_atemp_diff  temp_hum_interaction
0  0.24  0.2879           0.0479                0.1944
1  0.22  0.2727           0.0527                0.1760
2  0.22  0.2727           0.0527                0.1760
3  0.24  0.2879           0.0479                0.1800
4  0.24  0.2879           0.0479                0.1800


In [34]:
# Sort by time to ensure lags are correct
df = df.sort_values(by=['dteday', 'hr'])

# What was the demand 1 hour ago? 24 hours ago?
df['prv_hrs'] = df['cnt'].shift(1)
df['same_yes'] = df['cnt'].shift(24)

# Average demand over the last 3 hours
df['cnt_rolling_mean_3'] = df['cnt'].rolling(window=3).mean().shift(1)

print(df[['dteday', 'hr', 'cnt', 'prv_hrs', 'same_yes']].tail())

          dteday  hr  cnt  prv_hrs  same_yes
17374 2012-12-31  19  119    122.0     102.0
17375 2012-12-31  20   89    119.0      72.0
17376 2012-12-31  21   90     89.0      47.0
17377 2012-12-31  22   61     90.0      36.0
17378 2012-12-31  23   49     61.0      49.0


In [36]:
season_map = {1: 'spring', 2: 'summer', 3: 'fall', 4: 'winter'}
weather_map = {1: 'clear', 2: 'misty', 3: 'light_snow', 4: 'heavy_rain'}

df['season_label'] = df['season'].map(season_map)
df['weather_label'] = df['weathersit'].map(weather_map)

# Convert labels into binary (True/False) columns
df = pd.get_dummies(df, columns=['season_label', 'weather_label'], prefix=['season', 'weather'])

print(df.filter(like='season_').tail())


       season_fall  season_spring  season_summer  season_winter  season_fall  \
17374        False           True          False          False        False   
17375        False           True          False          False        False   
17376        False           True          False          False        False   
17377        False           True          False          False        False   
17378        False           True          False          False        False   

       season_spring  season_summer  season_winter  
17374           True          False          False  
17375           True          False          False  
17376           True          False          False  
17377           True          False          False  
17378           True          False          False  


In [37]:
df['cnt_log'] = np.log1p(df['cnt'])

print(df[['cnt', 'cnt_log']].head())

   cnt   cnt_log
0   16  2.833213
1   40  3.713572
2   32  3.496508
3   13  2.639057
4    1  0.693147


In [38]:
df['is_rush_hour'] = ((df['hr'].between(7, 9) | df['hr'].between(17, 19)) & (df['workingday'] == 1)).astype(int)

print(df[['hr', 'workingday', 'is_rush_hour']].sample(5))

       hr  workingday  is_rush_hour
7621    4           0             0
16309   7           0             0
15414  11           1             0
6764   10           1             0
14749  18           1             1


In [39]:
df['temp_bin'] = pd.cut(df['temp'], bins=[0, 0.3, 0.6, 1.0], labels=['Cold', 'Moderate', 'Hot'])

print(df[['temp', 'temp_bin']].head())

   temp temp_bin
0  0.24     Cold
1  0.22     Cold
2  0.22     Cold
3  0.24     Cold
4  0.24     Cold


In [40]:
df['weather_severity'] = df['weathersit'] * (1 + df['windspeed'])

print(df[['weathersit', 'windspeed', 'weather_severity']].head())

   weathersit  windspeed  weather_severity
0           1        0.0               1.0
1           1        0.0               1.0
2           1        0.0               1.0
3           1        0.0               1.0
4           1        0.0               1.0


In [41]:
df['heat_index_proxy'] = df['temp'] * df['hum']

print(df[['temp', 'hum', 'heat_index_proxy']].head())

   temp   hum  heat_index_proxy
0  0.24  0.81            0.1944
1  0.22  0.80            0.1760
2  0.22  0.80            0.1760
3  0.24  0.75            0.1800
4  0.24  0.75            0.1800


In [42]:
def get_day_part(hour):
    if 5 <= hour < 12: return 'Morning'
    elif 12 <= hour < 17: return 'Afternoon'
    elif 17 <= hour < 21: return 'Evening'
    else: return 'Night'

df['day_part'] = df['hr'].apply(get_day_part)

print(df[['hr', 'day_part']].head(10))

   hr day_part
0   0    Night
1   1    Night
2   2    Night
3   3    Night
4   4    Night
5   5  Morning
6   6  Morning
7   7  Morning
8   8  Morning
9   9  Morning
